# LoRA Rank Sensitivity Ablation — Colab A100

Tests split-QKV LoRA with r ∈ {4, 8, 16, 32} on EuroSAT s2_full, 3 seeds each.

- **12 runs total**, ~20 min each → **~4 GPU-hours**
- 可在单个 Colab A100 session 内跑完
- 结果保存到 Drive，格式与现有 `eurosat_results.json` 一致

**前置条件**：已跑过 `paper12_colab_run.ipynb` 的环境准备步骤（仓库 clone + 权重缓存）

## 0. 环境准备

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/paper12_results'
WEIGHTS_DIR = '/content/drive/MyDrive/paper12_weights'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results: {RESULTS_DIR}')

## 1. 克隆仓库 + 安装依赖

In [ ]:
%cd /content
!rm -rf AlphaEarth-System
!git clone https://github.com/zhouning/alphaearth-training-system.git AlphaEarth-System
%cd /content/AlphaEarth-System
!pip install -q -e '.[bench]'
!git log --oneline -3

## 2. 准备 Prithvi 权重

In [ ]:
import shutil

CACHED = f'{WEIGHTS_DIR}/Prithvi_100M.pt'
LOCAL  = '/content/AlphaEarth-System/data/weights/prithvi/Prithvi_100M.pt'
os.makedirs(os.path.dirname(LOCAL), exist_ok=True)

if os.path.exists(CACHED):
    print(f'Using cached weights: {CACHED}')
    shutil.copy(CACHED, LOCAL)
else:
    print('Downloading from HuggingFace...')
    !pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(repo_id='ibm-nasa-geospatial/Prithvi-100M', filename='Prithvi_100M.pt')
    shutil.copy(downloaded, LOCAL)
    shutil.copy(downloaded, CACHED)

print(f'Weights: {LOCAL} ({os.path.getsize(LOCAL)/1e6:.1f} MB)')

In [ ]:
# 注入 checkpoint 路径
import yaml, glob
for cfg_path in glob.glob('geoadapter/bench/configs/*.yaml'):
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    if 'prithvi' in cfg:
        cfg['prithvi']['checkpoint'] = LOCAL
        with open(cfg_path, 'w') as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        print(f'Patched: {cfg_path}')

## 3. 创建 Rank Ablation 配置

基于现有 `lora_ablation.yaml`，扩展为 r ∈ {4, 8, 16, 32}。

In [ ]:
import yaml

# 读取现有 lora_ablation 配置作为模板
base_cfg_path = 'geoadapter/bench/configs/lora_ablation.yaml'
with open(base_cfg_path) as f:
    base_cfg = yaml.safe_load(f)

print('Base config:')
print(yaml.dump(base_cfg, sort_keys=False))

In [ ]:
import copy

# 生成 rank ablation 配置：只跑 s2_full，4 个 rank × 3 seeds
rank_cfg = copy.deepcopy(base_cfg)

# 覆盖实验矩阵
rank_cfg['experiment']['methods'] = [
    {'name': 'lora_split_qkv', 'rank': 4},
    {'name': 'lora_split_qkv', 'rank': 8},
    {'name': 'lora_split_qkv', 'rank': 16},
    {'name': 'lora_split_qkv', 'rank': 32},
]
rank_cfg['experiment']['modalities'] = ['s2_full']
rank_cfg['experiment']['seeds'] = [42, 123, 456]

out_cfg = 'geoadapter/bench/configs/lora_rank_ablation.yaml'
with open(out_cfg, 'w') as f:
    yaml.safe_dump(rank_cfg, f, sort_keys=False)

print(f'Wrote: {out_cfg}')
print(f'Total experiments: 4 ranks × 3 seeds = 12')
print(yaml.dump(rank_cfg, sort_keys=False))

## 4. 运行 Rank Ablation (~4h)

12 个实验，每个 ~20 min。支持断点续跑（已完成的实验会跳过）。

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
out_json = f'{RESULTS_DIR}/lora_rank_ablation.json'

!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/lora_rank_ablation.yaml \
    --output {out_json} 2>&1 | tee {RESULTS_DIR}/lora_rank_ablation_{ts}.log

print(f'\n=> Results: {out_json}')

## 5. 结果分析

In [ ]:
import json
import pandas as pd

with open(out_json) as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(df[['method', 'modality', 'seed', 'rank', 'trainable_params', 'overall_accuracy', 'macro_f1']].to_string(index=False))

In [ ]:
# 按 rank 聚合
agg = df.groupby('rank')['overall_accuracy'].agg(['mean', 'std', 'count']).round(4)
agg.columns = ['OA_mean', 'OA_std', 'n_seeds']
print('\n=== LoRA Split-QKV Rank Sensitivity (EuroSAT s2_full) ===')
print(agg)
print(f'\nFor comparison: Houlsby OA = 0.821 ± 0.003, BitFit OA = 0.702 ± 0.002')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3.5))
ranks = agg.index.tolist()
means = agg['OA_mean'].tolist()
stds = agg['OA_std'].tolist()

ax.errorbar(ranks, means, yerr=stds, fmt='o-', capsize=4, color='C3', label='LoRA Split-QKV')
ax.axhline(y=0.821, color='C2', ls='--', label='Houlsby (d=64)')
ax.axhline(y=0.702, color='C1', ls=':', label='BitFit')
ax.axhline(y=0.657, color='C4', ls=':', alpha=0.6, label='Linear Probe')

ax.set_xlabel('LoRA rank r')
ax.set_ylabel('EuroSAT OA (s2_full)')
ax.set_xticks(ranks)
ax.set_ylim(0.6, 0.85)
ax.legend(fontsize=8)
ax.grid(True, ls=':', alpha=0.5)
fig.tight_layout()

fig_path = f'{RESULTS_DIR}/lora_rank_sensitivity.pdf'
fig.savefig(fig_path)
print(f'Figure saved: {fig_path}')
plt.show()

In [ ]:
# 合并到总 summary
import glob

all_results = []
for jf in sorted(glob.glob(f'{RESULTS_DIR}/*.json')):
    with open(jf) as f:
        rows = json.load(f)
    for r in rows:
        r['batch'] = os.path.basename(jf).replace('.json', '')
        all_results.append(r)

df_all = pd.DataFrame(all_results)
df_all.to_csv(f'{RESULTS_DIR}/summary.csv', index=False)
print(f'Updated summary: {RESULTS_DIR}/summary.csv ({len(df_all)} rows)')

## 6. 下一步

跑完后把 `lora_rank_ablation.json` 和 `lora_rank_sensitivity.pdf` 下载到本地 `paper12_results/`，然后：

```bash
# 本地重新生成论文图表（会自动读取新数据）
python paper12/scripts/make_figures.py
```

如果 rank 敏感性结果显示 r=16 或 r=32 仍低于 Houlsby，则进一步确认论文结论：
> "low-rank adaptation remains suboptimal even after the fused-QKV fix, regardless of rank."